In [3]:
from gensim.models.fasttext import load_facebook_vectors
from numpy import dot
from numpy.linalg import norm

print("Loading FastText vectors...")

ft = load_facebook_vectors("models/cc.en.300.bin")  # needs ~4–5 GB RAM

print("FastText vectors loaded.")

def cos_sim(w1, w2):
    v1, v2 = ft.get_vector(w1), ft.get_vector(w2)  # works for rare/OOV
    return float(dot(v1, v2) / (norm(v1) * norm(v2)))

Loading FastText vectors...
FastText vectors loaded.


In [5]:
print(cos_sim("king", "queen"))  # example usage
print(cos_sim("apple", "banana"))
print(cos_sim("computer", "banana"))

0.7068519592285156
0.5262576341629028
0.14826184511184692


In [50]:
import pandas as pd
phonosemantic_radicals = pd.read_excel("phonosemantic_radicals.xlsx")

# save to CSV
phonosemantic_radicals.to_csv("phonosemantic_radicals.csv", index=False)


def get_most_similar_radical(word):
    word_vector = ft.get_vector(word.lower())
    radicals = phonosemantic_radicals.to_dict(orient='records')
    similarities = []
    for radical in radicals:
        word = radical['Standalone']
        meaning = radical['Meaning']
        seed_words = radical['Seed Words'].split()
        words = [meaning] + seed_words
        similarity_scores = []
        for w in words:
            try:
                w_vector = ft.get_vector(w.lower())
                similarity = dot(word_vector, w_vector) / (norm(word_vector) * norm(w_vector))
                similarity_scores.append((w, similarity))
            except KeyError:
                continue
        if not similarity_scores:
            continue
        top_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)[:3]
        avg_similarity = sum(score for _, score in top_scores) / len(top_scores)
        max_similarity_words = [w for w, s in top_scores]
        similarities.append((word, avg_similarity, ", ".join(max_similarity_words)))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[0]

phonosemantic_radicals

,Standalone,Radical,Meaning,Seed Words
0,人,亻,man,uncle people crowd friend human
1,口,NaN,mouth,open phonetic onomatopoeia shout
2,水,氵,water,pour liquid juice slosh
3,冰,冫,ice,snow blizzard freeze cold
4,手,扌,hand,put fetch finger handle
5,言,讠,say,word speak declare text
6,木,NaN,tree,post table chair log
7,糸,纟,silk,loom textile sheet curtain
8,心,忄,heart,love sadness care angry
9,艸,艹,flora,bush vegetable leaf salad


In [51]:
text = "When in the course of human events it becomes necessary for one people to dissolve the political bands which have connected them with another and to assume among the powers of the earth the separate and equal station to which the laws of nature and of nature's God entitle them a decent respect to the opinions of mankind requires that they should declare the causes which impel them to the separation".split()
similarities = [get_most_similar_radical(word) for word in text]
for word, similarity in zip(text, similarities):
    print(word, similarity)

When ('日', np.float32(0.5903318), 'when, time, light')
in ('上', np.float32(0.31320724), 'up, above, high')
the ('又', np.float32(0.43674317), 'next, side, right')
course ('行', np.float32(0.4886808), 'course, path, walk')
of ('上', np.float32(0.23120844), 'up, high, above')
human ('人', np.float32(0.55013293), 'human, man, people')
events ('生', np.float32(0.23990345), 'life, live, youth')
it ('日', np.float32(0.3233976), 'when, time, sun')
becomes ('日', np.float32(0.23419462), 'when, hot, time')
necessary ('干', np.float32(0.32511798), 'achieve, gather, pursue')
for ('日', np.float32(0.30305108), 'time, when, light')
one ('又', np.float32(0.41721153), 'next, right, side')
people ('人', np.float32(0.57627684), 'people, man, crowd')
to ('行', np.float32(0.4247472), 'go, leave, arrive')
dissolve ('水', np.float32(0.3643142), 'pour, liquid, slosh')
the ('又', np.float32(0.43674317), 'next, side, right')
political ('人', np.float32(0.25049964), 'human, people, man')
bands ('金', np.float32(0.26731947), '